# 2. Information Protection and DLP

This notebook walks through Purview's end-to-end data protection pipeline:

1. **Classify** - figure out *what* data you have (sensitive info types, trainable classifiers).
2. **Label** - stamp each document with a sensitivity (Public -> Highly Confidential).
3. **Protect** - encrypt, watermark, restrict access via the label.
4. **Prevent leaks** - Data Loss Prevention (DLP) policies block risky sharing.
5. **Retain** - retention policies/labels decide how long items live.

## Setup

All Purview features in this notebook are **simulated** in Python - no Microsoft 365 tenant required.

1. `cd security-certs/sc-900/04-compliance-and-purview && uv sync`
2. Pick the **`.venv` kernel** from the VS Code kernel picker (top-right).
3. If it's missing: `Cmd+Shift+P` -> *Developer: Reload Window*.

---
## Step 1 - Data classification

Before you can protect data, you need to know *what kind* of data you have.

### Sensitive information types (SIT)

Pattern-based detectors built into Purview:

| SIT | Pattern | Example match |
|-----|---------|---------------|
| Credit card number | 16 digits with Luhn checksum | 4111-1111-1111-1111 |
| SSN (US) | XXX-XX-XXXX pattern | 123-45-6789 |
| Email address | user@domain | john@contoso.com |
| Passport number | Country-specific | Various |
| IBAN | Country code + check digits + account | DE89 3704 0044 0532 0130 00 |

### Trainable classifiers

ML models that recognize *types of content* instead of raw patterns:

- Resumes / CVs
- Source code
- Harassment language
- Financial statements

### Content Explorer vs. Activity Explorer

- **Content Explorer** - *where* does sensitive data live today? (e.g. "show all SharePoint files containing SSNs").
- **Activity Explorer** - *what are users doing* with it? (downloaded, shared, labeled, deleted).

Let's build a simulated Purview scanner:

In [1]:
import re

# Simulated sensitive information type (SIT) definitions.
def _luhn_check(num: str) -> bool:
    digits = [int(d) for d in num if d.isdigit()]
    odd_digits = digits[-1::-2]
    even_digits = digits[-2::-2]
    total = sum(odd_digits)
    for d in even_digits:
        total += sum(divmod(d * 2, 10))
    return total % 10 == 0

SENSITIVE_INFO_TYPES = [
    {
        'name': 'Credit Card Number',
        'pattern': r'\b(?:4[0-9]{12}(?:[0-9]{3})?|5[1-5][0-9]{14}|3[47][0-9]{13})\b',
        'validator': lambda m: _luhn_check(re.sub(r'[\s-]', '', m)),
        'confidence': 'high',
    },
    {
        'name': 'US Social Security Number',
        'pattern': r'\b\d{3}-\d{2}-\d{4}\b',
        'validator': lambda m: not m.startswith('000') and not m.startswith('666'),
        'confidence': 'high',
    },
    {
        'name': 'Email Address',
        'pattern': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
        'validator': lambda m: True,
        'confidence': 'medium',
    },
    {
        'name': 'IP Address',
        'pattern': r'\b(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\b',
        'validator': lambda m: True,
        'confidence': 'low',
    },
]

def scan_for_sensitive_data(text: str) -> list:
    findings = []
    for sit in SENSITIVE_INFO_TYPES:
        for match in re.finditer(sit['pattern'], text):
            value = match.group()
            if sit['validator'](value):
                masked = value[:4] + '***' + value[-4:] if len(value) > 8 else '***'
                findings.append({
                    'type': sit['name'],
                    'value': masked,
                    'position': match.start(),
                    'confidence': sit['confidence'],
                })
    return findings

DOCUMENTS = [
    {'name': 'employee_records.xlsx', 'content': 'Employee: John Doe, SSN: 123-45-6789, Email: john.doe@contoso.com, Phone: 555-0123'},
    {'name': 'payment_log.csv',       'content': 'Transaction: card 4111111111111111, amount $500. Merchant IP: 192.168.1.100.'},
    {'name': 'meeting_notes.docx',    'content': 'Discussed Q3 projections. Alice mentioned the project is on track. No action items.'},
]

print('=== Purview: Sensitive Information Scan ===\n')
for doc in DOCUMENTS:
    findings = scan_for_sensitive_data(doc['content'])
    flag = '[SENSITIVE]' if findings else '[CLEAN    ]'
    print(f'{flag} {doc["name"]}')
    for f in findings:
        print(f'   - {f["type"]} (confidence: {f["confidence"]}): {f["value"]}')
    print()

=== Purview: Sensitive Information Scan ===

[SENSITIVE] employee_records.xlsx
   - US Social Security Number (confidence: high): 123-***6789
   - Email Address (confidence: medium): john***.com

[SENSITIVE] payment_log.csv
   - Credit Card Number (confidence: high): 4111***1111
   - IP Address (confidence: low): 192.***.100

[CLEAN    ] meeting_notes.docx



---
## Step 2 - Sensitivity Labels

Once you know data is sensitive, you **label** it. Labels travel *with* the document (even if it's emailed outside) and enforce protection.

| Label | What it does | Example |
|-------|-------------|----------|
| **Public** | No restrictions | Press releases |
| **General** | No protection but marked | Internal memos |
| **Confidential** | Encryption + access control | Financial reports |
| **Highly Confidential** | Encryption + watermark + no forwarding | M&A docs, PII |

Labels can:

- **Encrypt** content (Azure RMS)
- **Add headers / footers / watermarks**
- **Restrict access** (only specific users/groups can open)
- **Auto-apply** when sensitive data is detected

### Label policies

Policies control *which* labels are available to *which* users, and set defaults:

- Publish labels to specific groups.
- Set a default label (e.g. "General" for all new docs).
- Require justification when downgrading (Confidential -> Public).

Below we auto-apply a sensitivity label based on what the scanner found:

In [2]:
# Auto-labeling rule: the highest-risk sensitive type wins.
def auto_label(findings: list) -> str:
    types = {f['type'] for f in findings}
    if {'Credit Card Number', 'US Social Security Number'} & types:
        return 'Highly Confidential'
    if types:
        return 'Confidential'
    return 'General'

LABEL_PROTECTIONS = {
    'Public':               {'encrypt': False, 'watermark': False, 'external_allowed': True},
    'General':              {'encrypt': False, 'watermark': False, 'external_allowed': True},
    'Confidential':         {'encrypt': True,  'watermark': True,  'external_allowed': False},
    'Highly Confidential':  {'encrypt': True,  'watermark': True,  'external_allowed': False},
}

print('=== Auto-labeling documents based on scan results ===\n')
labeled_docs = []
for doc in DOCUMENTS:
    findings = scan_for_sensitive_data(doc['content'])
    label = auto_label(findings)
    protection = LABEL_PROTECTIONS[label]
    labeled_docs.append({'name': doc['name'], 'findings': findings, 'label': label})
    print(f'{doc["name"]:<25} -> {label}')
    print(f'   encrypt: {protection["encrypt"]}, watermark: {protection["watermark"]}, share externally: {protection["external_allowed"]}')
    print()

=== Auto-labeling documents based on scan results ===

employee_records.xlsx     -> Highly Confidential
   encrypt: True, watermark: True, share externally: False

payment_log.csv           -> Highly Confidential
   encrypt: True, watermark: True, share externally: False

meeting_notes.docx        -> General
   encrypt: False, watermark: False, share externally: True



---
## Step 3 - Data Loss Prevention (DLP)

DLP prevents users from accidentally (or intentionally) sharing sensitive data. It watches:

- Exchange email
- SharePoint / OneDrive
- Teams chat and channels
- Endpoint devices (Windows / macOS)

A DLP policy has three parts:

1. **Conditions** - what triggers it (e.g. "document contains >= 1 credit card number").
2. **Actions** - what happens (block, warn, notify admin).
3. **User notifications / Policy Tips** - in-product coaching ("Heads up: this email has an SSN").

### Bad vs. Best: protecting a payroll spreadsheet

Let's compare what happens with and without labels + DLP turned on.

In [3]:
# BAD: no labels, no DLP. Anything goes.
print('=== BAD: no labels, no DLP ===')
payroll = 'Employees: Alice SSN 123-45-6789, Bob SSN 987-65-4321'
print(f'Doc content: {payroll}')
print('User action: forwards this to a personal Gmail account.')
print('Result: email leaves tenant. No block, no alert, no evidence.\n')

=== BAD: no labels, no DLP ===
Doc content: Employees: Alice SSN 123-45-6789, Bob SSN 987-65-4321
User action: forwards this to a personal Gmail account.
Result: email leaves tenant. No block, no alert, no evidence.



In [4]:
# BEST: scanner + auto-label + DLP policy block.
DLP_POLICIES = [
    {
        'name': 'Block external sharing of PII',
        'conditions': {'sensitive_types': ['US Social Security Number', 'Credit Card Number'], 'min_count': 1},
        'actions': {'internal_share': 'allow_with_warning', 'external_share': 'block', 'email_external': 'block'},
    },
    {
        'name': 'Block bulk PII movement',
        'conditions': {'sensitive_types': ['US Social Security Number'], 'min_count': 5},
        'actions': {'internal_share': 'block', 'external_share': 'block', 'email_external': 'block'},
    },
]

def evaluate_dlp(findings: list, action_type: str) -> list:
    results = []
    finding_types = [f['type'] for f in findings]
    for policy in DLP_POLICIES:
        matching = [t for t in policy['conditions']['sensitive_types'] if t in finding_types]
        count = len([f for f in findings if f['type'] in matching])
        if count >= policy['conditions']['min_count']:
            action = policy['actions'].get(action_type, 'allow')
            results.append({'policy': policy['name'], 'action': action, 'matched_types': matching, 'count': count})
    return results

print('=== BEST: scan -> auto-label -> DLP evaluation ===\n')
payroll_findings = scan_for_sensitive_data(payroll)
label = auto_label(payroll_findings)
print(f'Doc auto-labeled as: {label}')
print(f'Sensitive findings: {[f["type"] for f in payroll_findings]}')

scenarios = [
    ('Forward to personal Gmail', 'email_external'),
    ('Share with external contractor', 'external_share'),
    ('Send to internal #payroll channel', 'internal_share'),
]
action_icon = {'allow': 'ALLOW', 'allow_with_warning': 'WARN ', 'block': 'BLOCK'}
for name, action in scenarios:
    results = evaluate_dlp(payroll_findings, action)
    if not results:
        print(f'[ALLOW] {name}: no policy matched')
        continue
    for r in results:
        tag = action_icon[r['action']]
        print(f'[{tag}] {name}: policy "{r["policy"]}" matched {r["count"]} item(s)')
        if r['action'] == 'block':
            print('         policy tip: This content cannot be shared externally - contains PII.')

=== BEST: scan -> auto-label -> DLP evaluation ===

Doc auto-labeled as: Highly Confidential
Sensitive findings: ['US Social Security Number', 'US Social Security Number']
[BLOCK] Forward to personal Gmail: policy "Block external sharing of PII" matched 2 item(s)
         policy tip: This content cannot be shared externally - contains PII.
[BLOCK] Share with external contractor: policy "Block external sharing of PII" matched 2 item(s)
         policy tip: This content cannot be shared externally - contains PII.
[WARN ] Send to internal #payroll channel: policy "Block external sharing of PII" matched 2 item(s)


---
## Step 4 - Records Management and Retention

### Retention policies

Control how long content is **kept** and when it's **deleted**:

| Setting | Options |
|---------|----------|
| **Retain for** | X days / months / years |
| **After retention** | Delete automatically, or do nothing |
| **Apply to** | Exchange, SharePoint, OneDrive, Teams, Yammer |
| **Scope** | All users, specific users/groups, specific sites |

### Retention labels

More granular than policies - applied to *individual items*:

- Applied **manually** by users or **automatically** by rules.
- Can mark items as **records** (can't be modified or deleted).
- Can mark items as **regulatory records** (can't even be removed by admins).

### Key rule: retention wins over deletion

If a retention policy says "keep for 7 years" and a user deletes the file, it's preserved in a hidden location for 7 years.

**Exam tip**: retention *policies* are applied at the **location** level (all of SharePoint), retention *labels* are applied at the **item** level (specific document).

Below we simulate an email that a user *tries* to delete - but retention keeps it.

In [5]:
from datetime import date, timedelta

RETENTION_POLICY = {
    'name': 'Finance emails - 7 years',
    'location': 'Exchange',
    'scope': 'finance@contoso.com',
    'retain_years': 7,
    'after': 'delete',
}

email = {'id': 'mail-42', 'from': 'finance@contoso.com', 'created': date(2026, 4, 1), 'subject': 'Q1 invoice'}

print('=== Retention policy in action ===\n')
print(f'Policy: {RETENTION_POLICY}')
print(f'Email:  {email}\n')

print('User clicks Delete on Day 3...')
print('-> Visible inbox copy is removed.')
print('-> Hidden "Recoverable Items" copy is preserved until',
      email['created'] + timedelta(days=365 * RETENTION_POLICY['retain_years']))
print('-> After 7 years: automatic permanent deletion (because after="delete").')
print('\nLesson: retention wins over user deletion. Compliance officers love this; users are often surprised.')

=== Retention policy in action ===

Policy: {'name': 'Finance emails - 7 years', 'location': 'Exchange', 'scope': 'finance@contoso.com', 'retain_years': 7, 'after': 'delete'}
Email:  {'id': 'mail-42', 'from': 'finance@contoso.com', 'created': datetime.date(2026, 4, 1), 'subject': 'Q1 invoice'}

User clicks Delete on Day 3...
-> Visible inbox copy is removed.
-> Hidden "Recoverable Items" copy is preserved until 2033-03-30
-> After 7 years: automatic permanent deletion (because after="delete").

Lesson: retention wins over user deletion. Compliance officers love this; users are often surprised.


---
## End-to-end real-world scenario

**Scenario:** An HR analyst emails a spreadsheet called `new_hires_2026.xlsx` containing 12 SSNs to a recruiter at a staffing agency.

With Purview fully configured, here's what happens, step by step:

In [6]:
scenario_doc = {
    'name': 'new_hires_2026.xlsx',
    'content': ' '.join(f'Name{i} SSN: {100+i:03d}-{10+i:02d}-{1000+i:04d}' for i in range(12)),
}

print('[1] HR analyst attaches new_hires_2026.xlsx to an email to external recruiter.\n')

findings = scan_for_sensitive_data(scenario_doc['content'])
print(f'[2] Purview scanner finds {len(findings)} SSNs.\n')

label = auto_label(findings)
print(f'[3] Auto-label: {label} -> document is encrypted and watermarked.\n')

dlp_results = evaluate_dlp(findings, 'email_external')
print('[4] DLP evaluation:')
for r in dlp_results:
    print(f'    - policy "{r["policy"]}" ({r["count"]} matches) -> {r["action"].upper()}')
print()

print('[5] User sees a Policy Tip: "This email contains SSNs and cannot leave the organization."')
print('[6] Email is blocked; alert sent to the compliance team; incident logged for audit.')
print('[7] HR analyst opens a ticket to request a proper secure-transfer workflow.')

[1] HR analyst attaches new_hires_2026.xlsx to an email to external recruiter.

[2] Purview scanner finds 12 SSNs.

[3] Auto-label: Highly Confidential -> document is encrypted and watermarked.

[4] DLP evaluation:
    - policy "Block external sharing of PII" (12 matches) -> BLOCK
    - policy "Block bulk PII movement" (12 matches) -> BLOCK

[5] User sees a Policy Tip: "This email contains SSNs and cannot leave the organization."
[6] Email is blocked; alert sent to the compliance team; incident logged for audit.
[7] HR analyst opens a ticket to request a proper secure-transfer workflow.


---
## Summary

| Concept | Key fact |
|---------|----------|
| **Sensitive info types** | Pattern-based detection (SSN, credit cards, etc.) |
| **Trainable classifiers** | ML-based content classification |
| **Content Explorer** | Where does sensitive data exist? |
| **Activity Explorer** | What are users doing with sensitive data? |
| **Sensitivity labels** | Classify + protect (encrypt, watermark, restrict) |
| **DLP** | Prevent sharing of sensitive data (block, warn, notify) |
| **Retention policies** | Keep/delete content by age, applied at location level |
| **Retention labels** | Keep/delete individual items, can make records |

**Next**: [Notebook 3 - Insider Risk and eDiscovery](03_insider_risk_and_ediscovery.ipynb)